In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

Loading the datasets

In [ ]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

In [ ]:
print("Train Dataset :", train.shape)
print("Public Test Dataset :", public_test.shape)
print("Private Test Dataset :", private_test.shape)

In [ ]:
train.head()

In [ ]:
public_test.head()

In [ ]:
train.columns

In [ ]:
train.info()

In [ ]:
train.describe()


In [ ]:
train.isnull().sum()

In [ ]:
(train.isnull().sum() / len(train)) * 100

In [ ]:
train["Converted"].value_counts()

In [ ]:
train["Converted"].value_counts(normalize=True) * 100

In [ ]:
sns.countplot(x="Converted", data=train)
plt.title("Converted vs Not Converted")
plt.show()

In [ ]:
num_cols = [
    "Age",
    "Income",
    "Pages_Viewed",
    "Products_Viewed",
    "Time_On_Site",
    "Previous_Purchases"
]
cat_cols = [
    "City_Tier",
    "Device_Type",
    "Traffic_Source",
    "Browser_Version",
    "Campaign_Code"
]

In [ ]:
for col in cat_cols:
    print("\nColumn:", col)
    print(train[col].unique())

In [ ]:
train_data = train.copy()
public_data = public_test.copy()
private_data = private_test.copy()

Observation:
Dataset contains missing values in Age, Income and Time_On_Site.
Target variable is slightly imbalanced.
Both numerical and categorical features are present.
User_ID is only an identifier and may not help in prediction.

Exploratory Data Analysis (EDA)

In [ ]:
for col in num_cols:    
    plt.figure(figsize=(5,3))
    sns.histplot(train[col], kde=True)
    plt.title(col)
    plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize=(5,3))
    sns.boxplot(x=train[col])
    plt.title(col)
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
corr = train[num_cols].corr()
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
city_conversion = train.groupby(
    "City_Tier"
)["Converted"].mean()
print(city_conversion)

In [ ]:
city_conversion.plot(kind="bar")
plt.ylabel("Conversion Rate")
plt.title("Conversion Rate by City Tier")
plt.show()

In [ ]:
device_conversion = train.groupby(
    "Device_Type"
)["Converted"].mean()
print(device_conversion)

In [ ]:
device_conversion.plot(kind="bar")
plt.ylabel("Conversion Rate")
plt.title("Conversion Rate by Device")
plt.show()

In [ ]:
traffic_conversion = train.groupby(
    "Traffic_Source"
)["Converted"].mean()
print(traffic_conversion)

In [ ]:
traffic_conversion.plot(kind="bar")
plt.ylabel("Conversion Rate")
plt.title("Conversion Rate by Traffic Source")
plt.show()

In [ ]:
discount_conversion = train.groupby(
    "Discount_Seen"
)["Converted"].mean()
print(discount_conversion)

In [ ]:
sns.barplot(
    x="Discount_Seen",
    y="Converted",
    data=train
)
plt.title("Discount Seen vs Conversion")
plt.show()

In [ ]:
train.groupby(
    "Converted"
)["Income"].mean()

In [ ]:
train.groupby(
    "Converted"
)["Time_On_Site"].mean()

In [ ]:
sns.boxplot(
    x="Converted",
    y="Pages_Viewed",
    data=train
)
plt.show()

In [ ]:
sns.boxplot(
    x="Converted",
    y="Products_Viewed",
    data=train
)
plt.show()

Observations:
1. Dataset contains missing values in Age, Income and Time_On_Site.
2. Converted users generally spend more time on the website.
3. Users who view more products tend to convert more often.
4. Conversion rate differs across traffic sources and device types.
5. Discount visibility appears to have some effect on conversion.
6. No extremely strong correlations were observed among numerical features.

Data Preprocessing

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data["Age"].fillna(
    train_data["Age"].median(),
    inplace=True
)
train_data["Income"].fillna(
    train_data["Income"].median(),
    inplace=True
)
train_data["Time_On_Site"].fillna(
    train_data["Time_On_Site"].median(),
    inplace=True
)

In [ ]:
public_data["Age"].fillna(
    train_data["Age"].median(),
    inplace=True
)
public_data["Income"].fillna(
    train_data["Income"].median(),
    inplace=True
)
public_data["Time_On_Site"].fillna(
    train_data["Time_On_Site"].median(),
    inplace=True
)

In [ ]:
private_data["Age"].fillna(
    train_data["Age"].median(),
    inplace=True
)
private_data["Income"].fillna(
    train_data["Income"].median(),
    inplace=True
)
private_data["Time_On_Site"].fillna(
    train_data["Time_On_Site"].median(),
    inplace=True
)

In [ ]:
print(train_data.isnull().sum().sum())
print(public_data.isnull().sum().sum())
print(private_data.isnull().sum().sum())

In [ ]:
cat_cols = [
    "City_Tier",
    "Device_Type",
    "Traffic_Source",
    "Browser_Version",
    "Campaign_Code"
]

In [ ]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        train_data[col],
        public_data[col],
        private_data[col]
    ])

    le.fit(combined)

    train_data[col] = le.transform(train_data[col])

    public_data[col] = le.transform(public_data[col])

    private_data[col] = le.transform(private_data[col])

In [ ]:
X = train_data.drop(
    ["User_ID", "Converted"],
    axis=1
)
y = train_data["Converted"]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
num_cols = [
    "Age",
    "Income",
    "Pages_Viewed",
    "Products_Viewed",
    "Time_On_Site",
    "Previous_Purchases"
]

In [ ]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)
X_valid[num_cols] = scaler.transform(
    X_valid[num_cols]
)

In [ ]:
X_public = public_data.drop(
    ["User_ID", "Converted"],
    axis=1
)
y_public = public_data["Converted"]

In [ ]:
X_public[num_cols] = scaler.transform(
    X_public[num_cols]
)

In [ ]:
X_private = private_data.drop(
    ["User_ID"],
    axis=1
)

In [ ]:
X_private[num_cols] = scaler.transform(
    X_private[num_cols]
)

In [ ]:
print(X_train.shape)
print(X_valid.shape)
print(X_public.shape)
print(X_private.shape)

Observations:

1. Missing values were present in Age, Income and Time_On_Site.
2. Median imputation was used to handle missing values.
3. Categorical features were converted into numerical form using Label Encoding.
4. Numerical features were standardized before applying Logistic Regression.
5. User_ID was removed since it does not contribute to prediction.

Logistic Regression Model

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train, y_train)

In [ ]:
train_pred = lr_model.predict(X_train)
print(
    "Training Accuracy:",
    accuracy_score(y_train, train_pred)
)

In [ ]:
valid_pred = lr_model.predict(X_valid)

In [ ]:
print(
    "Validation Accuracy:",
    accuracy_score(y_valid, valid_pred)
)

In [ ]:
f1 = f1_score(
    y_valid,
    valid_pred
)

print("Validation F1 Score:", f1)

In [ ]:
print(
    classification_report(
        y_valid,
        valid_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_valid,
    valid_pred
)
print(cm)

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
valid_prob = lr_model.predict_proba(X_valid)[:,1]
auc = roc_auc_score(
    y_valid,
    valid_prob
)
print("ROC AUC Score:", auc)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_model.coef_[0]
})
feature_importance.sort_values(
    by="Coefficient",
    ascending=False
)

In [ ]:
feature_importance.sort_values(
    by="Coefficient",
    ascending=False
).head(10)

In [ ]:
feature_importance.sort_values(
    by="Coefficient"
).head(10)

In [ ]:
public_pred = lr_model.predict(X_public)

In [ ]:
print(
    "Public Test Accuracy:",
    accuracy_score(
        y_public,
        public_pred
    )
)

In [ ]:
public_f1 = f1_score(
    y_public,
    public_pred
)

print(
    "Public Test F1 Score:",
    public_f1
)

In [ ]:
print(
    classification_report(
        y_public,
        public_pred
    )
)

In [ ]:
train_acc = accuracy_score(
    y_train,
    train_pred
)
valid_acc = accuracy_score(
    y_valid,
    valid_pred
)
print("Train Accuracy :", train_acc)
print("Validation Accuracy :", valid_acc)

Model Observations:
1. Logistic Regression was used as the baseline classification model.
2. Numerical features were standardized before training.
3. Model performance was evaluated using Accuracy, ROC-AUC and F1 Score.
4. Public test performance was also checked to estimate generalization ability.
5. Logistic Regression provides interpretable coefficients which help understand feature influence.

Prediction and Submission

In [ ]:
private_pred = lr_model.predict(X_private)

In [ ]:
print(private_pred[:10])

In [ ]:
submission = pd.DataFrame({
    "User_ID": private_data["User_ID"],
    "Converted": private_pred
})

In [ ]:
submission.head()

In [ ]:
print(submission.shape)

In [ ]:
submission["Converted"].value_counts()

In [ ]:
submission["Converted"].value_counts(normalize=True) * 100

In [ ]:
submission.to_csv(
    "submission.csv",
    index=False
)
print("Submission file created successfully")

In [ ]:
saved_file = pd.read_csv("submission.csv")
saved_file.head()

In [ ]:
saved_file.info()

In [ ]:
public_accuracy = accuracy_score(
    y_public,
    public_pred
)
print("Public Test Accuracy:", public_accuracy)